In [ ]:
from IPython.display import Image
from IPython.core.display import HTML 
from IPython.display import IFrame

# DATATIE Luento 5

Tämä luentomuistion ensimmäisen version valmisteli Arho Suominen. Vuonna 2025 DATATIEtä luennoi Jukka Huhtamäki.




# Ohjaamaton oppiminen

Opintojaksolla on tähän mennessä keskitytty ohjattuun oppimiseen. Ohjatussa oppimisessa tavoitteena on ennustaa piirre y, kun annettuna on x. Asetelma ei kuitenkaan sovi kaikkiin data-analyysitehtäviin, joten tarvetta on myös lähestymistavoille, jossa datajoukossa S = {x1, . . . , xn} ei ole erikseen määriteltyä ennustettavaa piirrettä.


![Unsupervised workflow](image/Unsupervised-Learning-Workflow-73_W640.jpg)


![Unsupervised learning workflow](image/unsupervised-learning-workflow.webp)



## Ohjaamattoman oppimisen otteita

* [Ostoskorianalyysi](https://pbpython.com/market-basket-analysis.html)
* Ryvästäminen 
* Aihemallinnus

## Ryvästäminen

Ryvästämisellä tarkoitetaan sitä, että syötteen pisteet jaetaan kokonaisuuksiin eli ryppäisiin eli klustereihin siten, että kukin rypäs sijaitseen mahdollisimman pienellä alueella, mutta ryppäät sijaitsevat mahdollisimman kaukana toisistaan.

Käytännön sovellus on siis sellainen, jossa käyttöösi on annettu tietojoukko, jossa jokaisella tapauksella on joukko yhteisiä ominaisuuksia. Tietojoukossa ei kuitenkaan ole tietoa siitä, mihin luokkaan tai luokkiin syöte kuuluu. Luokkien tunteminen on keskeistä ohjatulle algoritmille, kuten tukivektorikoneille ([Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html)), joka oppii ennustamaan luokat saamansa oppimisdataan pohjautuen.

Ohjaamattomassa ryvästämisessä on keskeistä löytää keskenään erilaisia klustereita. K-Means -algoritmit ovat hyvä esimerkki ryvästämisestä. Tavoitteena on siis arvioida mihin keskuspisteeseen syöte kuuluu. K-Means -algoritmit pyrkivät löytämään keskuspisteet määrittelemällä syötteen pisteitä klustereihin, jotka perustuvat nykyisiin keskuspisteisiin. Keskuspisteiden sijaintia päivitetään laskennan edetessä vaiheittain.

In [ ]:
!pip install mglearn

In [ ]:
import numpy as np
import pandas as pd
import pylab as pl
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn import preprocessing
import matplotlib.pyplot as plt
import holoviews as hv
from holoviews import opts
# https://github.com/amueller/mglearn
# from src import mglearn
# import mglearn
hv.extension('bokeh')
%matplotlib inline

Käytetään Kaggle:stä löytyvää [elokuva-aineistoa](https://www.kaggle.com/rounakbanik/the-movies-dataset#movies_metadata.csv). Aineisto on siitä hyvä, että se vaatii aika vähän esitöitä, piirteet ovat paikallaan ja suurimmaksi osaksi aineistossa ei ole merkittäviä puutteita. Tämähän ei suinkaan ole yleistä.

Vai onko aineisto "hyvä"? Miten se selvitettäisiin?

In [ ]:
df = pd.read_csv("data/the-movies-dataset/movies_metadata.csv", low_memory=False)
df.head()

In [ ]:
df.info()

Datassa on 24 saraketta, joista useat ovat kategorisia. KMeans-analyysi ei pysty ottamaan kategorisia muuttujia, koska niiden etäisyysmitta ei ole merkityksellinen. Jos haluaisimme käyttää kategorisia muuttujia, meidän tulisi soveltaa K-Modes-analyysiä tai ratkaisua, joka yhdistää KModes- ja KMeans-analyysin.

Suositeltavaa luettavaa kategorisen datan ryvästämisestä esimerkiksi Hen (2006) [K-Mode -ryvästämistä käsittelevästä artikkelista](https://arxiv.org/ftp/cs/papers/0603/0603120.pdf). K-modes -menetelmän soveltaminen onnistuu esimerkiksi [kmodes-paketilla](https://github.com/nicodv/kmodes).

Keskitytään tässä kuitenkin KMeans-ryvästämiseen ja luodaan uusi rakenne, jossa mukana ovat vain numeerisia arvoja sisältävät sarakkeet. Pidetään mukana myös otsikko ja kuvaus mukana jatkoa varten.

In [ ]:
dfNum = df[['budget','popularity','revenue','runtime','vote_average','vote_count','title','overview']]
dfNum.head()

Aineisto on aika siisti, mutta tarkistetaan nyt vielä ainakin puuttuvat arvot ja muut keskeiset ongelmakohdat.

In [ ]:
dfNum.isnull().sum()

In [ ]:
tietueita = dfNum.shape
tietueita

Tarkastellaan tyhjiä soluja sisältäviä rivejä, jos voidaan havaita puuttuuko yksittäisistä soluista tieto vai yleensä koko rivi tyhjä. Ei tunnu mielekkäältä täydentää keskiarvolla soluja jos koko havainnon tiedot tulee syötettäväksi. Tässä aineistossa näyttää siltä, että "runtime" on jäänyt usealta elokuvalta täydentämättä. Voisiko tämän täydentää?

In [ ]:
dfNum[dfNum.isnull().any(axis=1)]

Epäilyttävältä aineistossa vaikuttaa se, että niissä soluissa joista puuttuu "runtime", myös budjetti ja tuotto ovat nolla. Poistetaan siis nämä tietueet ja otetaan mukaan vain ne tietueet joissa kaikki arvot ovat saatavilla.

In [ ]:
dfNum=dfNum.dropna()
print("Poistettuja tietueita oli: "+str(tietueita[0]-dfNum.shape[0])+", jäljellä "+str(dfNum.shape[0]))

Poistetaan KMeans-analyysistä tekstimuotoiset muuttujat.

In [ ]:
dfTitleOverview=dfNum[['title', 'overview']]
dfNum =dfNum.drop(['title', 'overview'], axis=1)

Tarkistetaan muuttujatyypit

In [ ]:
dfNum = dfNum.astype(float)
dfNum.info()

Pandas kuvailee dataa helposti describe() toiminnolla. Voiko tästä tehdä jotain huomioita?

In [ ]:
dfNum.describe()

Jos laitetaan kuvaileva informaation histogrammeihin, voidaanko tästä erottaa jotain mitä aineistossa pitäisi huomioida?

In [ ]:
hist = dfNum.hist()

Poistetaan vielä budjetti- ja liikevaihtotiedot, joiden kaikissa kvartiileissa arvot ovat edelleen 0. On syytä epäillä, että merkittävässä osassa nämä tiedot ovat puutteelliset.

In [ ]:
dfNum=dfNum.drop(['budget','revenue'], axis=1)

Normalisoidaan muuttujat. Normalisoinnissa on kuitenkin useita vaihtoehtoja. Kannattaa lukea Ben [Alex Green blogiteksti normalisointitavoista](http://benalexkeen.com/feature-scaling-with-scikit-learn/). Täältä löytää esimerkiksi tiedon siitä, miksi [Min Max Scaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) ei välttämättä ole paras normalisointi mekanismi. Mikä voisi toimia paremmin?

In [ ]:
minmax = preprocessing.MinMaxScaler().fit_transform(dfNum)
dfNumNorm = pd.DataFrame(minmax, index=dfNum.index, columns=dfNum.columns)
dfNumNorm.head()

In [ ]:
dfNumNorm.describe()

### KMeans - miten se toimii

Miten KMeans toimii? Valitaan klustereiden määrä K. Klustereille valitaaan keskispisteet ja lasketaan jokaisen havainnon etäisyys keskipisteistä. Sijoitetaan jokainen havainto klusteriin sen perusteella, mikä keskipiste on lähimpänä. Lasketaan uudelleen keskipisteiden paikat perustuen syntyneisiin klustereihin. Toistetaan kunnes keskipisteet eivät enää muutu.

Tunnistettuja ongelmia on esimerkiksi se että 1) poikkeavat havainnot (outlier) vaikuttavat tulokseen merkittävästi ja 2) aineiston järjestys voi vaikuttaa lopputulokseen, joten analyysin stabiliteetti tulee varmistaa.

![KMeans klusterointi](https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/K-means_convergence.gif/440px-K-means_convergence.gif)

### Miten arvioida klusteiden määrä?

Ei ole mitään yksiselitteistä tapaa valita klustereiden määrää. Asiaa kannattaa tarkastella pohtimalla kyseessä olevaa ongelmakenttää. Esimerkiksi, kuinka monta klusteria voisi olla elokuva-aineistossa?

Yksi lähestymistapa aineiston klustereiden määrän valintaan on kyynärpääperiaate (engl. elbow curve). Lähestymistapa toimii niin, että lasketaan neliösumma (klusterin keskipisteen ja havaintojen etäisyyksien neliösumma) eri klustereiden määrillä. Mikäli klustereiden määrä on sama kuin datapisteiden määrä, havaintojen kuvaaja menee nollaan. Etsimme kuitenkin pistettä, jossa klustereiden määrän lisääminen tuo enää vain pienen hyödyn.

![Elbow diagram](https://upload.wikimedia.org/wikipedia/commons/c/cd/DataClustering_ElbowCriterion.JPG)

In [ ]:
# Valitaan laskettavien klustereiden määrä. Mikä olisi järkevä klustereiden määrä maksimissaan.
# Onko 20 liikaa?
c = range(1, 20)
# Sovitetaan for-silmukassa KMeans kaikille c:n arvoille
kmeans = [KMeans(n_clusters=i) for i in c]
score = [kmeans[i].fit(dfNumNorm).score(dfNumNorm) for i in range(len(kmeans))]

In [ ]:
hv.Curve(score)

Valitaan kolme klusteria, jonka jälkeen näyttää olevan vain vähän hyötyä siitä että lisäämme klustereita.

In [ ]:
kmeans = KMeans(n_clusters=3)
kmeans.fit(dfNumNorm)

In [ ]:
dfNum['cluster'] = kmeans.labels_

In [ ]:
dfNum.head(25)

Katsotaan miten klusterit jakautuvat aineistossa. Tämä voi antaa suuntaviivoja siitä onko klusterointi mielekäs. Olennaistahan on ymmärtää mistä aineistosta teimme jaon.

In [ ]:
bars = hv.Bars(dfNum['cluster'].value_counts())
bars

Tarkastellaan aineistoa vielä sen perusteella, miten elokuvan keskiarvo ja suosio voisivat liittyä klustereihin.

In [ ]:
scatter = hv.Scatter(dfNum, kdims=[ 'vote_average','popularity'], vdims=['cluster']).groupby(['cluster'])
scatter.overlay('cluster')

Mitä voimme arvioida kuvaajasta?

In [ ]:
dfNum.head(25)

## Luonnollisen kielen analyysi ja aihemallinnus

Aihemallinnus (topic modeling) on menetelmä dokumenttien kokoelmassa esiintyvien abstraktien "aiheiden" löytämiseksi. Aihemallinnusta käytetään dokumenttikokoelmassa olevien piilotettujen semanttisten rakenteiden löytämiseksi. Lähtökohtana on että dokumenteista on löydettävissä piirteitä erilaisista aiheista ja kokoelmassa nämä aiheet esiintyvät enemmän kuin yhdessä dokumentissa.

Lisää aihemallinnuksesta voi lukea [David M. Blein artikkeleista](http://www.cs.columbia.edu/~blei/topicmodeling.html). 
Yau, Porter, Newman ja Suominen ([2014](https://link.springer.com/article/10.1007/s11192-014-1321-8)) vertailivat eri aihemallinnusmenetelmien suorituskykyä. 
Tiedoksi myös aihemallinnusta hyödyntävä suomenkielinen artikkeli [Aihemallinnus hybridin mediatapahtuman ja merkitysten kierron tutkimuksessa](https://doi.org/10.23983/mv.91078) (Toivanen, Huhtamäki, Valaskivi ja Tikka, 2020).

![LDA](image/IntroToLDA.png)

Kuvalähde:  Sterbak ([2018](https://www.depends-on-the-definition.com/understanding-text-data-with-topic-models/))

In [ ]:
IFrame("https://en.wikipedia.org/wiki/Document-term_matrix", width=1000, height=400)

In [ ]:
IFrame("https://reader.elsevier.com/reader/sd/pii/S0040162516303651?token=5E5D0D4AF151658E30202F6AB22A4D30538C9A6C0C74AA16555CF19859D280BF09CA4A034137E11111F6A67077D5C911", width=1000, height=800)

Käytetään jälleen NLTK-kirjastoa tekstin esikäsittelyyn. Tätä nykyä [spaCy](https://spacy.io/) on se hyödyllisin vaihtoehto. 

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
# Hukkasanojen poisto
stopword = set(stopwords.words("english"))
translator = str.maketrans('', '', string.punctuation)

Käytetään samaa aineistoa jota käytimme aiemmin ja hyödynnetään tällä kertaan tekstiaineistoa. Arvioidaan minkälaisia piileviä aihepiirejä sisällöstä muodostuu.

In [ ]:
kuvaukset=dfTitleOverview['overview'].tolist()

Kuten numeeriset muuttujat, myös luonnollinen kieli vaatii prosessointia ennen analyysiä. Tämä tulisi kuitenkin tehdä niin että emme poista kaikkea sisältöä.

In [ ]:
kuvauksetPutsattu=[]
for kuvaus in kuvaukset:
        # poistetaan sanat jotka sisältävät numeron.
        kuvaus = " ".join([x for x in kuvaus.split(" ") if not x.isdigit()]) #terms consisting of only numbers removed
        # poistetaan välimerkit
        kuvaus= kuvaus.translate(translator) 
        # Pidetään vain sanat jotka ovat pidempiä kuin kolmen merkkiä
        kuvaus = " ".join(sana for sana in kuvaus.split() if len(sana)>3) #Keep words that are longer than 3
        # Poistetaan vielä kaikki sanat jotka sisältyvät hukkasanalistaan
        kuvaus = ' '.join([sana for sana in kuvaus.split() if sana not in stopword])
        kuvauksetPutsattu.append(kuvaus)

In [ ]:
print("Tekstistä:\n"+kuvaukset[0]+ "\n\ntuli prosessin jälkeen:\n" + kuvauksetPutsattu[0])

Käytetään aihemallin luomiseen suosittua [Gensim-kirjastoa](https://radimrehurek.com/gensim/). Gensim on jo pitkään käytössä ollut tekstin semanttisen analyysin kirjasto. [BERTopic](https://maartengr.github.io/BERTopic/index.html) tarjoaa kiinnostavan ja helposti lähestyttävän vaihtoehdon. 

Yleisin käyttötarkoitus on etsiä semanttisesti samankaltaisia dokumentteja. Paketti tarjoaa mahdollisuudeen tekstin esikäsittelylle, semanttisen tekstin muuttamiseksi vektoriksi sekä mallintamiseen [LSI](https://en.wikipedia.org/wiki/Latent_semantic_analysis)- ja [LDA]-(https://en.wikipedia.org/wiki/Latent_Dirichlet_allocation) algoritmien kanssa.

In [ ]:
IFrame("https://radimrehurek.com/gensim/auto_examples/core/run_corpora_and_vector_spaces.html#from-strings-to-vectors", width=1000, height=800)

In [ ]:
from gensim.models import Phrases
from gensim.models import Word2Vec
from gensim import corpora, models, similarities, matutils
from gensim.models import hdpmodel, ldamodel, TfidfModel

# Cleaning and creating corpus
dictionary = corpora.Dictionary(line.lower().split() for line in kuvauksetPutsattu)
limit = 1
once_ids = [tokenid for tokenid, docfreq in dictionary.dfs.items() if docfreq <= limit]
dictionary.filter_tokens(once_ids) # remove stop words and words that appear only once'
dictionary.compactify() # remove gaps in id sequence after words that were removed
dictionary.save("corpora.dict")

class MyCorpus(object):
    def __iter__(self):
        for line in kuvauksetPutsattu:
            # assume there's one document per line, tokens separated by whitespace
            yield dictionary.doc2bow(line.lower().split())
mem_friendly_corpus = MyCorpus() # doesn't load the corpus into memory!
corpora.MmCorpus.serialize('corpus.mm', mem_friendly_corpus)
#corpora.BleiCorpus.serialize(CorpusBlei, mem_friendly_corpus, id2word=dictionary)

dictionary = corpora.Dictionary.load("corpora.dict")

analysisCorpus = corpora.MmCorpus('corpus.mm')

Samoin kuin KMeans-analyysissä, on meidän pyrittävä arvioimaan mielekäs määrä piileviä osajoukkoja aineistossa. [Chang et al.](http://users.umiacs.umd.edu/~jbg/docs/nips2009-rtl.pdf) kuvaavat hyvin, että piilevien luokkien määrä on todennäköisesti helpoiten arvioitavissa havainnoimalla tuloksia. Tarjolla on kuitenkin myös muita mahdollisuuksia, kuten KL-divergenssi ([Arun et al. 2010](https://link.springer.com/chapter/10.1007/978-3-642-13657-3_43))

In [ ]:
# Estimate KL Divergence to input
# Hard coded to range 1 -100 step 1 
# Returns a list of values
import scipy.stats as stats

def sym_kl(p,q):
    return np.sum([stats.entropy(p,q),stats.entropy(q,p)])

def arun(corpus,dictionary):
    
    l = np.array([sum(cnt for _, cnt in doc) for doc in corpus])
    kl = []
    for i in range(1,20,1):
        lda = models.ldamodel.LdaModel(corpus=corpus,
            id2word=dictionary,num_topics=i)
        m1 = lda.expElogbeta
        U,cm1,V = np.linalg.svd(m1)
        # Document-topic matrix
        lda_topics = lda[analysisCorpus]
        m2 = matutils.corpus2dense(lda_topics, lda.num_topics).transpose()
        cm2 = l.dot(m2)
        cm2 = cm2 + 0.0001
        cm2norm = np.linalg.norm(l)
        cm2 = cm2/cm2norm
        kl.append(sym_kl(cm1,cm2))
    return kl

In [ ]:
kl = arun(analysisCorpus, dictionary)

In [ ]:
hv.Curve(kl)

KL-divergenssin perusteella valitsemme 3 piilevää teemaa.

In [ ]:
# Run LDA
n_topics=3
n_docs=len(kuvauksetPutsattu)

lda = models.ldamodel.LdaModel(analysisCorpus, id2word=dictionary, num_topics=n_topics)
lda_corpus = lda[analysisCorpus]
# lda.save(LDA_model)

Piilevät teemat kuvautuvat sana-teema -matriisina, jossa jokaisella sanalla on todennäköisyys kuulua jokaiseen piilevään teemaan. Oheinen koodin tulostaa jokaisesta teemasta 10 todennäköisintä sanaa.

In [ ]:
temp_terms = lda.show_topics(num_topics=int(n_topics), num_words=11, log=False, formatted=False)
index = 0
topTerms={}
for topic in range(n_topics):
    for wordn in range(11):
        try:
            topTerms['Topic '+str(topic+1)].append(temp_terms[topic][1][wordn][0])
        except KeyError:
            topTerms['Topic '+str(topic+1)]=[temp_terms[topic][1][wordn][0]]
ldaTermsDf=pd.DataFrame(topTerms)
ldaTermsDf

Olemmeko luoneet oheisella jaottelulla mielekkäitä elokuvaluokkia? Tämä on ehkä hieman kyseenalaista, mutta palataan takaisin aikaisempaan katsomaa miltä luokat näyttävät isosta aineistosta.

In [ ]:
IFrame("https://rajapinta.co/2017/07/08/varovaisuutta-aihemallinnuksen-kanssa/", width=1000, height=800)

In [ ]:
hardCluster=[]
docProb=[]
for y in range(0, n_docs):
    prob=lda[lda_corpus[y]]
    hc=max(lda[lda_corpus[y]],key=lambda item:item[1])
    hardCluster.append(hc[0])
    probList=[]
    for i in prob:
             probList.append(i[1])
    docProb.append(probList)
probDf=pd.DataFrame(docProb, columns=['Topic 1','Topic 2','Topic 3'])
probDf.head()

Topic modeling -algoritmi on pehmeän luokittelun (soft classification) algoritmi, jossa jokainen termi ja dokumentti saa todennäköisyyden kaikkiin teemoihin. Tämä vaatii osaltaan oikeanlaista lähestymistavan valintaa. Jos esimerkiksi valitsemme vain korkeimman todennäköisyyden luokan menetämme paljon informaatiota.

In [ ]:
dfNum['topics']=hardCluster

In [ ]:
dfNum['title']=dfTitleOverview['title']

Onko luokitteluissa järkeä?

In [ ]:
dfNum.head(25)

In [ ]:
dfNum.pivot_table(index='cluster', columns='topics', aggfunc={'topics':len}, fill_value=0)

Zemaityte ja muut ([2024](https://doi.org/10.1371/journal.pone.0297404)) soveltavat luovasti tekstin semanttista analyysia elokuvafestivaaleja käsittelevässä tutkimuksessaan. Myös [tutkimusaineisto](https://doi.org/10.6084/m9.figshare.22682794.v1) on julkaistu.

Käydään vielä katsomassa [Googlen prosessikuvaa](https://cloud.google.com/blog/products/gcp/google-patents-public-datasets-connecting-public-paid-and-private-patent-data), joka kertoo hyvin sen miten dataa tyypillisesti koostetaan useista lähteistä.

## Yhteenveto

Vedetään lopuksi yhteen keskeiset havainnot ohjaamattomasta oppimisesta

1. Ohjaamattoman oppisen menetelmiä ovat esimerkiksi ryvästäminen, aihemallinnus, ostoskorianalyysi ja verkostoanalyysi
1. Myös ohjaamattomassa oppimisessa tapahtuu ohjaamista. Esimerkiksi klustereiden määrän valinta KMeans-algoritmia sovellettaessa tai aiheiden määrä aihemallinnuksessa. 
3. Aineiston laadun merkitys: garbage in, garbage out -periaate pätee
4. Etenkin aihemallinnus on hyvin subjektiivista  
5. Aineiston eksploratiivisen analytiikan rooli on keskeinen

Mitä opimme tänään?

1. Myös ohjaamattomassa oppimisessa analyytikon rooli on keskeinen, aktiivinen. Esimerkiksi klustereiden määrä on valittava.
2. Etenkin aihemallinnuksessa subjektiivisuus ja tulosten laadullinen tulkinta korostuu.
3. Aineiston laadu on ehdoton edellytys analyysin onnistumiselle myös ohjaamattomassa oppimisessa. Roskaa sisään, roskaa ulos.